# Information Extraction from PDF Sections

This notebook tests the information extraction pipeline.

The pipeline performs the following steps:

1. Load the useful sections from the CSV file.
2. Extract text and tables from each section.
3. Use PaddleOCR as a fallback when native PDF extraction is insufficient.
4. Send each section to the LLM using `extract_with_llm`.
5. Run the LLM extraction for all sections in parallel.
6. Inspect and evaluate the extracted information.

## 0. Imports

In [1]:
from pathlib import Path
import pandas as pd
import sys

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.information_extraction import (
    load_sections,
    extract_page_content,
    process_section,
    process_sections_parallel
)

from src.ocr.ocr_pipeline import ocr_pipeline

c:\Users\intel\industry\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\intel\.paddlex\official_models\PP-DocLayout_plus-L`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\intel\.paddlex\official_models\SLANet`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\intel\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [31]:
OUTPUT_DIR = PROJECT_ROOT / "results"

PDF_PATH = PROJECT_ROOT / "data" / "raw" / "AUSTCOLD.pdf"
SECTIONS_CSV = PROJECT_ROOT / "data" / "processed" / "resolved_sections.csv"

print("PDF:", PDF_PATH)
print("CSV:", SECTIONS_CSV)

PDF: c:\Users\intel\Downloads\industrial-equipment-data-extraction\data\raw\AUSTCOLD.pdf
CSV: c:\Users\intel\Downloads\industrial-equipment-data-extraction\data\processed\resolved_sections.csv


## 1. Load sections

In [5]:
sections = load_sections(SECTIONS_CSV)

print(f"Number of sections: {len(sections)}")

Number of sections: 15


In [6]:
sections_df = pd.DataFrame(sections)
sections_df

,title,start_page,end_page
0,3.02 Compressor Data Sheet,107,115
1,3.03 Compressor motor data sheet,116,154
2,3.04 Oil Pump Data Sheet,155,156
3,3.05 Oil Pump Motor Data Sheet,157,159
4,3.06 Oil Cooler Data Sheet,160,165
5,3.07 Oil separator data sheet,166,169
6,3.08 Receiver data sheet,170,170
7,3.09 Purger data sheet,171,173
8,3.10 Economiser Data Sheet,174,178
9,3.11 Air Cooled Condenser data sheet,179,182


## 2. Extract page content

In [7]:
test_page = sections[0]["start_page"]

page_result = extract_page_content(
    pdf_path=PDF_PATH,
    page_number=test_page,
    ocr_pipeline=ocr_pipeline,
    min_text_length=50,
    dpi=300
)

In [8]:
page_result

{'page': 107,
 'text': "NUMÉRO DU DOCUMENT A5114-TDS-18\nRÉVISION\n01\n02\n03\n04\n05\nDATE\nBY\nMH\nRÉV/APPR\nCS\n  N° DE JOB\nN° ITEM\n  PAGE\n1\nDE\n9\nN° RÉQ.\n1\nAPPLICABLE À :\n    PROPOS°\n      ACHAT\nCONFORME À L'EXÉCUTION\n2\nPOUR\nUNITÉ\n3\nSITE\nN° DE SÉRIE\n4\nSERVICE\nN° REQUIS\n5\nFABRICANT\nMODÈLE\nPILOTE  (6.1)\n6\nREMARQUE :INFORMATIONS À COMPLETER PAR L'ACHETEUR\nPAR LE FABRICANT\n7\nCONDITIONS DE FONCTIONNEMENT\n8\nCAS D'EXPLOITATION N°\n9\nTOUTES LES DONNÉES SONT SUR UNE BASE UNITAIRE\n10\n11\nPOINT CERTIFIÉ (          )   \n12\nGAZ TRAITÉS (VOIR ÉGALEMENT PAGE 2\n13\nCapacité requise Nm3/h (1,013 bar & 0 °C) (SEC) (3.46 & 3.60)\n14\nDÉBIT MASSE, KG/HR-  (MOUILLÉ)(SEC)\n15\nCONDITIONS D'ENTRÉE :\nBRIDE ORIFICE D'ENTRÉE COM\nCONNEXION DU CLIENT\n16\nPRESSION - absolue (bar) \n17\nTEMPÉRATURE (°C)\n18\nHUMIDITÉ RELATIVE (%)\n19\nMASSE MOLÉCULAIRE (M)\n20\nCp/Cv (K1) OU (KAVG) (5.1.15.4)\n21\nCOMPRESSIBILITÉ (Z1) OU (ZAVG) (5.1.15.5)\n22\nDÉBIT VOLUME ORIFICE D'ENTRÉE

## 3. Process one complete section

In [9]:
test_section = sections[0]

print(test_section)

{'title': '3.02 Compressor Data Sheet', 'start_page': 107, 'end_page': 115}


In [10]:
test_result = process_section(
    pdf_path=PDF_PATH,
    section=test_section,
    ocr_pipeline=ocr_pipeline,
    #llm=llm,
    dpi=300,
    min_text_length=50
)

Processing section: 3.02 Compressor Data Sheet (pages 107-115)


In [11]:
test_result

{'method': 'llm',
 'called': True,
 'result': {'family': {'value': None, 'confidence': 0.0, 'page': None},
  'asset_name': {'value': 'Compresseur de type rotatif à déplacement positif',
   'confidence': 1.0,
   'page': 107},
  'reference': {'value': '255-V-100A/B/C/D-C-01',
   'confidence': 1.0,
   'page': 107},
  'power': {'value': '610 kW', 'confidence': 1.0, 'page': 111},
  'outlier': {'value': None, 'confidence': 0.0, 'page': None},
  'manufacturer': {'value': 'HOWDEN', 'confidence': 1.0, 'page': 107},
  'asset_diagram': {'value': None, 'confidence': 0.0, 'page': None}}}

## 4. Parallel LLM Extraction

Each useful section is processed independently.

The LLM extraction is executed in parallel using a `ThreadPoolExecutor`.

The number of simultaneous LLM requests is controlled by `max_workers`.

For example:

- `max_workers=1` → sequential execution
- `max_workers=2` → 2 simultaneous requests
- `max_workers=4` → 4 simultaneous requests

We start with 4 workers to avoid sending too many requests simultaneously.

In [12]:
results = process_sections_parallel(
    pdf_path=PDF_PATH,
    csv_path=SECTIONS_CSV,
    max_workers=4,
    dpi=300,
    min_text_length=50
)

Found 15 sections.

Starting parallel processing with 4 workers...

Processing section: 3.02 Compressor Data Sheet (pages 107-115)
Processing section: 3.03 Compressor motor data sheet (pages 116-154)
Processing section: 3.04 Oil Pump Data Sheet (pages 155-156)
Processing section: 3.05 Oil Pump Motor Data Sheet (pages 157-159)
Native extraction insufficient on page 157. Using OCR...
[1/5] Loaded: PDF 'AUSTCOLD.pdf' (Page 157, 300 DPI)
[2/5] Preprocessing completed
Processing section: 3.06 Oil Cooler Data Sheet (pages 160-165)✗ Failed: 3.05 Oil Pump Motor Data Sheet → (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc:118)


Native extraction insufficient on page 116. Using OCR...
[1/5] Loaded: PDF 'AUSTCOLD.pdf' (Page 116, 300 DPI)
[2/5] Preprocessing completed
Processing section: 3.07 Oil separator data sheet (pages 166-169)
✗ Failed: 3.03 Compr

In [13]:
print("Number of sections:", len(sections))
print("Number of results:", len(results))

Number of sections: 15
Number of results: 15


In [14]:
failed_results = [
    result
    for result in results
    if result.get("error") is not None
]

print(f"Failed sections: {len(failed_results)}")

Failed sections: 12


In [24]:
results_df = pd.DataFrame([
    {
        "title": result["title"],
        "start_page": result["start_page"],
        "end_page": result["end_page"],
        #"num_tables": len(result["tables"]),
        #"methods": ", ".join(set(result["extraction_methods"])),
        "success": result.get("error") is None,
        "error": result.get("error")
    }
    for result in results
])

results_df

,title,start_page,end_page,success,error
0,3.02 Compressor Data Sheet,107,115,True,NaN
1,3.03 Compressor motor data sheet,116,154,False,(Unimplemented) ConvertPirAttribute2RuntimeAtt...
2,3.04 Oil Pump Data Sheet,155,156,True,NaN
3,3.05 Oil Pump Motor Data Sheet,157,159,False,(Unimplemented) ConvertPirAttribute2RuntimeAtt...
4,3.06 Oil Cooler Data Sheet,160,165,False,(Unimplemented) ConvertPirAttribute2RuntimeAtt...
5,3.07 Oil separator data sheet,166,169,False,(Unimplemented) ConvertPirAttribute2RuntimeAtt...
6,3.08 Receiver data sheet,170,170,True,NaN
7,3.09 Purger data sheet,171,173,False,(Unimplemented) ConvertPirAttribute2RuntimeAtt...
8,3.10 Economiser Data Sheet,174,178,False,(Unimplemented) ConvertPirAttribute2RuntimeAtt...
9,3.11 Air Cooled Condenser data sheet,179,182,False,(Unimplemented) ConvertPirAttribute2RuntimeAtt...


In [26]:
for result in results:
    print()
    print(f"SECTION: {result['title']}")
    print("-" * 100)

    print(result["llm_result"])


SECTION: 3.02 Compressor Data Sheet
----------------------------------------------------------------------------------------------------
{'method': 'llm', 'called': True, 'result': {'family': {'value': None, 'confidence': 0.0, 'page': None}, 'asset_name': {'value': 'Compresseur de réfrigération', 'confidence': 1.0, 'page': 107}, 'reference': {'value': '255-V-100A/B/C/D-C-01', 'confidence': 1.0, 'page': 107}, 'power': {'value': '610 kW', 'confidence': 1.0, 'page': 111}, 'outlier': {'value': None, 'confidence': 0.0, 'page': None}, 'manufacturer': {'value': 'Howden', 'confidence': 1.0, 'page': 107}, 'asset_diagram': {'value': None, 'confidence': 0.0, 'page': None}}}

SECTION: 3.03 Compressor motor data sheet
----------------------------------------------------------------------------------------------------
None

SECTION: 3.04 Oil Pump Data Sheet
----------------------------------------------------------------------------------------------------
{'method': 'llm', 'called': True, 'result'

In [29]:
print(results[0]["llm_result"])

{'method': 'llm', 'called': True, 'result': {'family': {'value': None, 'confidence': 0.0, 'page': None}, 'asset_name': {'value': 'Compresseur de réfrigération', 'confidence': 1.0, 'page': 107}, 'reference': {'value': '255-V-100A/B/C/D-C-01', 'confidence': 1.0, 'page': 107}, 'power': {'value': '610 kW', 'confidence': 1.0, 'page': 111}, 'outlier': {'value': None, 'confidence': 0.0, 'page': None}, 'manufacturer': {'value': 'Howden', 'confidence': 1.0, 'page': 107}, 'asset_diagram': {'value': None, 'confidence': 0.0, 'page': None}}}


In [33]:
results_df.to_json(OUTPUT_DIR / "extracted_sections.json", orient="records", indent=2)